# Assignment 3 – Parte 1: Visualización con Plotly

**Grupo 7 · Temporada: enero–mayo de 2024**

## Pregunta

**¿Los departamentos con mayor lluvia acumulada son también los que registran más declaratorias de emergencia durante enero–mayo de 2024?**

**Fuente de datos:** `tabla_final.csv` elaborado por el Grupo 7 en el Assignment 2 y reutilizado en `assignment_3/datos/dataset.csv`.


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

BASE = Path(".") if Path("datos/dataset.csv").exists() else Path("assignment_3")
RUTA = BASE / "datos" / "dataset.csv"

df = pd.read_csv(RUTA, dtype={"ubigeo": "string"})

print("Dimensiones:", df.shape)
display(df.head())


Dimensiones: (25, 9)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,455.0,0,4,2
1,02,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3,5,5
2,03,Apurímac,Abancay,-13.63426,-72.88380,670.6,2,7,4
3,04,Arequipa,Arequipa,-16.39899,-71.53747,235.3,1,7,6
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,467.5,1,7,5


## Variables utilizadas

Para responder la pregunta se usarán principalmente:

- `departamento`: unidad geográfica.
- `lluvia_total_mm`: lluvia acumulada en milímetros.
- `declaratorias`: número de declaratorias de emergencia.
- `prorrogas`: número de prórrogas.
- `dias_lluvia_fuerte`: días con lluvia fuerte.

El análisis es **descriptivo**: busca identificar patrones y contrastes, no establecer causalidad.


In [2]:
print("Departamentos con más declaratorias:")
display(
    df[["departamento", "declaratorias", "lluvia_total_mm"]]
    .sort_values(["declaratorias", "lluvia_total_mm"], ascending=[False, False])
    .head(10)
)

print("\nDepartamentos con mayor lluvia acumulada:")
display(
    df[["departamento", "lluvia_total_mm", "declaratorias"]]
    .sort_values("lluvia_total_mm", ascending=False)
    .head(5)
)

print("\nDepartamentos con menor lluvia acumulada:")
display(
    df[["departamento", "lluvia_total_mm", "declaratorias"]]
    .sort_values("lluvia_total_mm")
    .head(5)
)


Departamentos con más declaratorias:


,departamento,declaratorias,lluvia_total_mm
8,Huancavelica,7,819.2
2,Apurímac,7,670.6
7,Cusco,7,601.0
4,Ayacucho,7,467.5
3,Arequipa,7,235.3
9,Huánuco,6,416.7
14,Lima,6,12.2
11,Junín,5,1276.9
1,Áncash,5,1058.3
20,Puno,5,728.6



Departamentos con mayor lluvia acumulada:


,departamento,lluvia_total_mm,declaratorias
15,Loreto,1361.8,3
11,Junín,1276.9,5
16,Madre de Dios,1149.5,3
1,Áncash,1058.3,5
24,Ucayali,887.4,4



Departamentos con menor lluvia acumulada:


,departamento,lluvia_total_mm,declaratorias
14,Lima,12.2,6
6,Callao,17.3,0
10,Ica,19.3,4
13,Lambayeque,25.6,0
12,La Libertad,26.3,2


## Gráfico 1 – Declaratorias por departamento


In [3]:
orden = df.sort_values(["declaratorias", "lluvia_total_mm"], ascending=[True, True])

fig_barras = px.bar(
    orden,
    x="declaratorias",
    y="departamento",
    orientation="h",
    text="declaratorias",
    labels={
        "declaratorias": "Número de declaratorias",
        "departamento": "Departamento"
    },
    title=(
        "Cinco departamentos alcanzan el máximo de 7 declaratorias"
        "<br><sup>Fuente: Assignment 2 del Grupo 7, temporada ene–may 2024</sup>"
    )
)

fig_barras.update_traces(textposition="outside")
fig_barras.update_layout(template="plotly_white", height=700, xaxis=dict(dtick=1))
fig_barras.show()

salida_html = BASE / "grafico.html"
fig_barras.write_html(salida_html)
print("Gráfico HTML exportado en:", salida_html)


Gráfico HTML exportado en: grafico.html


## Gráfico 2 – Lluvia acumulada vs. declaratorias


In [4]:
correlacion = df["lluvia_total_mm"].corr(df["declaratorias"])

fig_disp = px.scatter(
    df,
    x="lluvia_total_mm",
    y="declaratorias",
    hover_name="departamento",
    hover_data={
        "dias_lluvia_fuerte": True,
        "prorrogas": True,
        "lluvia_total_mm": ":.1f",
        "declaratorias": True
    },
    labels={
        "lluvia_total_mm": "Lluvia acumulada (mm)",
        "declaratorias": "Número de declaratorias"
    },
    title=(
        f"La lluvia acumulada tiene una relación débil con las declaratorias (r = {correlacion:.2f})"
        "<br><sup>Fuente: Assignment 2 del Grupo 7, temporada ene–may 2024</sup>"
    )
)

fig_disp.update_traces(marker=dict(size=11, opacity=0.8))
fig_disp.update_layout(template="plotly_white", height=550)
fig_disp.show()

print(f"Correlación lluvia–declaratorias: {correlacion:.2f}")


Correlación lluvia–declaratorias: 0.20


## Gráfico 3 – Distribución de la lluvia acumulada


In [5]:
fig_hist = px.histogram(
    df,
    x="lluvia_total_mm",
    nbins=8,
    labels={"lluvia_total_mm": "Lluvia acumulada (mm)"},
    title=(
        "Pocos departamentos concentran los niveles más altos de lluvia acumulada"
        "<br><sup>Fuente: Assignment 2 del Grupo 7, temporada ene–may 2024</sup>"
    )
)

fig_hist.update_layout(
    template="plotly_white",
    height=500,
    yaxis_title="Número de departamentos"
)
fig_hist.show()


## Verificación e interpretación

Los valores extremos mostrados en las tablas coinciden con los gráficos:

- **Huancavelica, Apurímac, Cusco, Ayacucho y Arequipa** registran el máximo de **7 declaratorias**.
- **Loreto, Junín y Madre de Dios** presentan los mayores acumulados de lluvia.
- Estos grupos no coinciden plenamente, por lo que **más lluvia no se traduce automáticamente en más declaratorias**.
- La correlación es baja (aprox. **0.20**), lo cual respalda una relación lineal débil en esta muestra.

**Conclusión.** Para la temporada analizada, la cantidad de lluvia acumulada por sí sola no explica el número de declaratorias de emergencia. El análisis es descriptivo y no implica causalidad.

**Limitación.** La lluvia utilizada corresponde a la capital departamental y no representa necesariamente la variabilidad de todo el departamento.
